# Used Cars Cost Prediction (India)
## 1. Business problem and data source
**Problem:** buyers and sellers of used cars in India struggle to know whether a listing is fairly priced. We build a model that predicts a fair price range and flags listings as over- or under-priced.
**Data:** CarDekho used-car listings (Kaggle, 'Vehicle dataset'). The target is the listing price in INR. Limitations: no city column, the data ends in 2020, and prices are listing prices, not confirmed transaction prices.

### 1.1 Load the data

In [1]:
# Cell 1: imports and load
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 50)
RAW = "../data/raw/Car details v3.csv"
df_raw = pd.read_csv(RAW)
print("Raw shape:", df_raw.shape)
df_raw.head()

Raw shape: (8128, 13)


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


### 1.2 Data audit
Before modelling, we check data quality. Duplicates matter most: if identical rows land in both train and test, the model memorises them and the score is inflated.

In [2]:
# Cell 2: audit table
audit = {
    "rows_raw": len(df_raw),
    "exact_duplicates": int(df_raw.duplicated().sum()),
    "rows_after_dedupe": len(df_raw.drop_duplicates()),
    "year_min": int(df_raw.year.min()), "year_max": int(df_raw.year.max()),
    "price_min": int(df_raw.selling_price.min()), "price_max": int(df_raw.selling_price.max()),
    "median_price": int(df_raw.selling_price.median()),
}
print(pd.Series(audit))
print("\nMissing values:\n", df_raw.isna().sum()[lambda s: s > 0])

rows_raw                 8128
exact_duplicates         1202
rows_after_dedupe        6926
year_min                 1983
year_max                 2020
price_min               29999
price_max            10000000
median_price           450000
dtype: int64

Missing values:
 mileage      221
engine       221
max_power    215
torque       222
seats        221
dtype: int64


### 1.3 Are the missing values the same rows?
If the gaps in mileage, engine, max_power, torque and seats are the same rows, they are systematic (a scraping gap), not random, and this decides how we treat them.

In [3]:
# Cell 3: missing-value pattern
spec_cols = ["mileage", "engine", "max_power", "torque", "seats"]
miss = df_raw[spec_cols].isna()
print("Rows missing ANY spec:", miss.any(axis=1).sum())
print("Rows missing ALL specs:", miss.all(axis=1).sum())

Rows missing ANY spec: 222
Rows missing ALL specs: 215


### 1.4 Listings per car model
Later we validate by holding out whole car models, so the model is tested on cars it has never seen. This check shows whether we have enough listings per model for that.

In [4]:
# Cell 4: listings per model
df_raw["brand"] = df_raw["name"].str.split().str[0]
df_raw["model_key"] = df_raw["name"].str.split().str[:2].str.join(" ")
per_model = df_raw.drop_duplicates().model_key.value_counts()
print("Distinct brand+model:", per_model.size)
print(per_model.describe())
print("Models with < 5 listings:", (per_model < 5).sum())

Distinct brand+model: 210
count    210.000000
mean      32.980952
std       68.986717
min        1.000000
25%        3.000000
50%        8.000000
75%       32.000000
max      670.000000
Name: count, dtype: float64
Models with < 5 listings: 77


### 1.5 Findings
(leave this cell as a placeholder; I will write the findings after I see the outputs)